In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import transformer_lens

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
def getDevice():
    if torch.cuda.is_available(): #nvidia/runpod
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps") #apple silicon
    else:
        return torch.device("cpu")
    
DEVICE = getDevice()

device(type='cpu')

In [ ]:
# model_name = "Qwen/Qwen1.5-1.8B-Chat"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map=DEVICE)

In [ ]:
# messages = [
#     {"role": "system", "content": "You are a helpful assistant."},
#     {"role": "user", "content": "Write a short poem about the ocean."}
# ]

# text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# inputs = tokenizer(text, return_tensors="pt").to(model.device)

In [ ]:
# output_ids = model.generate(**inputs, max_new_tokens=100)
# response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# print(response)

The vast and endless blue,
A canvas of infinite depth,
A world of secrets untold,
Where creatures swim with ease.

From tiny plankton to massive whales,
Their songs fill the air with melody,
As the waves crash upon the shore,
Creating a symphony in harmony.

The salty breeze whispers secrets,
Of tides that ebb and flow,
Of storms that rage and calm,
And how they shape the land we call home.

The horizon stretches on forevermore,
As stars shine above like


# TransformerLens

In [24]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tl_model = transformer_lens.HookedTransformer.from_pretrained(
    model_name,
    device=DEVICE,
    dtype=torch.float16,
    move_to_device=True
)


Loaded pretrained model Qwen/Qwen1.5-1.8B-Chat into HookedTransformer


In [25]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short poem about the ocean."}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

tokens = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

In [26]:
generated = tokens.clone()

for _ in range(100):
    with torch.no_grad():
        logits = tl_model(generated)
    next_token = torch.argmax(logits[:, -1, :], dim=-1)
    generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)
    if next_token.item() == tokenizer.eos_token_id:
        break

response = tokenizer.decode(generated[0][tokens.shape[1]:], skip_special_tokens=True)
print(response)

KeyboardInterrupt: 